In [ ]:
!pip install transformers datasets evaluate jiwer soundfile librosa -q
!pip install spacy nltk -q
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 117.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
LANGUAGE  = "English"
TASK      = "transcribe"
HUB_REPO  = "MFouni/whisper-small-tedlium-v3"

In [ ]:
import torch
import evaluate
import numpy as np
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    pipeline,
)

In [ ]:
processor = WhisperProcessor.from_pretrained(HUB_REPO)
model = WhisperForConditionalGeneration.from_pretrained(HUB_REPO)
model = model.to("cuda")
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [ ]:
test = load_dataset(
    "AudioLLMs/tedlium3_test",
    split="test",
)

test = test.remove_columns(
    [col for col in test.column_names if col not in ["context", "answer"]]
)

test = test.cast_column("context", Audio(sampling_rate=16_000))

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/301M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1142 [00:00<?, ? examples/s]

In [ ]:
def prepare_dataset(batch, processor):
    audio_arrays   = [a["array"] for a in batch["context"]]
    sampling_rates = [a["sampling_rate"] for a in batch["context"]]

    batch["input_features"] = [
        processor.feature_extractor(a, sampling_rate=sr).input_features[0]
        for a, sr in zip(audio_arrays, sampling_rates)
    ]
    batch["labels"] = [
        processor.tokenizer(text).input_ids
        for text in batch["answer"]
    ]
    return batch

print("Preprocessing test set...")
test_processed = test.map(
    prepare_dataset,
    remove_columns=test.column_names,
    fn_kwargs={"processor": processor},
    batched=True,
    batch_size=16,
    num_proc=1,
    desc="Preprocessing test"
)
test_processed = test_processed.with_format("torch")

Preprocessing test set...


Preprocessing test:   0%|          | 0/1142 [00:00<?, ? examples/s]

In [ ]:
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
wer_metric    = evaluate.load("wer")

dataloader = DataLoader(
    test_processed,
    batch_size=8,
    collate_fn=data_collator,
)

all_preds  = []
all_labels = []

print("Running inference on test set...")
with torch.no_grad():
    for batch in dataloader:
        input_features = batch["input_features"].to("cuda")
        labels         = batch["labels"]

        # Generate predictions
        predicted_ids = model.generate(
            input_features,
            language=LANGUAGE.lower(),
            task=TASK,
        )

        # Decode predictions
        pred_str = processor.tokenizer.batch_decode(
            predicted_ids, skip_special_tokens=True
        )

        # Decode labels
        labels[labels == -100] = processor.tokenizer.pad_token_id
        label_str = processor.tokenizer.batch_decode(
            labels, skip_special_tokens=True
        )

        all_preds.extend(pred_str)
        all_labels.extend(label_str)

wer = wer_metric.compute(predictions=all_preds, references=all_labels)
print(f"\n{'='*50}")
print(f"Test WER: {round(wer * 100, 2)}%")
print(f"{'='*50}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Running inference on test set...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.



Test WER: 8.91%


In [ ]:
print("Loading raw test samples for qualitative analysis...")
raw_test = load_dataset(
    "AudioLLMs/tedlium3_test",
    split="test[:10]",
)

pipe = pipeline(
    "automatic-speech-recognition",
    model=HUB_REPO,
    device=0,
)

Loading raw test samples for qualitative analysis...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
for i, (pred, label) in enumerate(zip(all_preds[:10], all_labels[:10])):
    sample_wer = wer_metric.compute(
        predictions=[pred],
        references=[label]
    )
    print(f"\nSample {i+1}")
    print(f"Ground truth : {label}")
    print(f"Prediction   : {pred}")
    print(f"Sample WER   : {round(sample_wer * 100, 2)}%")
    print(f"{'='*60}")


Sample 1
Ground truth : i 'd like to share with you a discovery that i made a few months ago while writing an article for italian wired i always keep my thesaurus handy whenever i'm writing anything but
Prediction   : i would like to share with you a discovery that i made a few months ago while writing an article for italian wired i always keep my bazaar as handy whenever i am writing anything but
Sample WER   : 14.71%

Sample 2
Ground truth : i 'd already finished editing the piece and i realized that i had never once in my life looked up the word disabled to see what i 'd find let me read you the entry
Prediction   : i had already finished editing the piece and i realized that i had never once in my life looked up the word disabled to see what i would find let me read you the entry
Sample WER   : 5.71%

Sample 3
Ground truth : disabled adjective crippled helpless useless wrecked
Prediction   : disabled adjectives crippled helpless useless wrecked
Sample WER   : 16.67%

Sample 4
Grou

In [ ]:
import spacy
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    results = {}

    # ── 1. Sentence Segmentation ──────────────────────────────
    sentences = sent_tokenize(text)
    results["sentences"] = sentences

    # ── 2. Tokenization ───────────────────────────────────────
    tokens = word_tokenize(text.lower())
    results["tokens"] = tokens

    # ── 3. Stopword Removal ───────────────────────────────────
    filtered_tokens = [
        t for t in tokens
        if t.isalpha() and t not in stop_words
    ]
    results["filtered_tokens"] = filtered_tokens

    # ── 4. POS Tagging ────────────────────────────────────────
    pos_tags = pos_tag(filtered_tokens)
    results["pos_tags"] = pos_tags

    return results

In [ ]:
preprocessed = []

for i, text in enumerate(all_preds[:10]):
    print(f"\n{'='*60}")
    print(f"Sample {i+1}")
    print(f"{'='*60}")

    result = preprocess_text(text)
    preprocessed.append(result)

    print(f"Original text:\n  {text}\n")

    print(f"Sentences ({len(result['sentences'])}):")
    for s in result["sentences"]:
        print(f"  - {s}")

    print(f"\nTokens ({len(result['tokens'])}):")
    print(f"  {result['tokens']}")

    print(f"\nFiltered tokens ({len(result['filtered_tokens'])}) (no stopwords):")
    print(f"  {result['filtered_tokens']}")

    print(f"\nPOS Tags:")
    for token, tag in result["pos_tags"]:
        print(f"  {token:20s} → {tag}")


Sample 1
Original text:
  i would like to share with you a discovery that i made a few months ago while writing an article for italian wired i always keep my bazaar as handy whenever i am writing anything but

Sentences (1):
  - i would like to share with you a discovery that i made a few months ago while writing an article for italian wired i always keep my bazaar as handy whenever i am writing anything but

Tokens (36):
  ['i', 'would', 'like', 'to', 'share', 'with', 'you', 'a', 'discovery', 'that', 'i', 'made', 'a', 'few', 'months', 'ago', 'while', 'writing', 'an', 'article', 'for', 'italian', 'wired', 'i', 'always', 'keep', 'my', 'bazaar', 'as', 'handy', 'whenever', 'i', 'am', 'writing', 'anything', 'but']

Filtered tokens (18) (no stopwords):
  ['would', 'like', 'share', 'discovery', 'made', 'months', 'ago', 'writing', 'article', 'italian', 'wired', 'always', 'keep', 'bazaar', 'handy', 'whenever', 'writing', 'anything']

POS Tags:
  would                → MD
  like               